In [ ]:
!git clone https://github.com/RUDRA-DIXIT/FAS-XAI-ZDA.git


In [ ]:
%cd FAS-XAI-ZDA


In [ ]:
!ls -R


In [ ]:
print("===== MODEL ARCHITECTURE =====")
!cat src/models/fas_bilstm.py

print("\n===== FEATURE ATTENTION RANKING =====")
!cat experiments/phase1_reproduced/feature_attention_ranking.json

print("\n===== FEATURE SELECTION =====")
!cat experiments/phase1_reproduced/feature_selection.json


In [ ]:
print("===== PREPROCESSING =====")
!cat src/data/preprocess.py

print("\n===== DATA LOADER =====")
!cat src/data/load_nslkdd.py

print("\n===== REQUIREMENTS =====")
!cat requirements.txt


In [ ]:
!find data -maxdepth 3 -type f


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls -lh /content/drive/MyDrive/NSL-KDD/

In [ ]:
!sed -n '1,240p' src/data/zero_day_split.py

In [ ]:
!python -m src.data.zero_day_split \
    --train /content/drive/MyDrive/NSL-KDD/KDDTrain+.txt \
    --out data/processed/primary_split \
    --seed 42

In [ ]:
!python -m src.data.preprocess \
    --split-dir data/processed/primary_split \
    --out data/processed/arrays_with_level

In [ ]:
import numpy as np
import joblib

base = "data/processed/arrays_with_level"

for name in ["train", "validation", "test_primary"]:
    data = np.load(f"{base}/{name}.npz")
    print(f"\n{name}")
    print("X shape:", data["X"].shape)
    print("y shape:", data["y"].shape)

bundle = joblib.load(f"{base}/preprocessor.joblib")
print("\nNumber of features:", len(bundle["features"]))
print("Includes level:", bundle["include_level"])

In [ ]:
import tensorflow as tf
import numpy as np

MODEL_PATH = (
    "experiments/phase1_reproduced/"
    "01_full_features_with_attention/best_model.keras"
)

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

print("Model loaded successfully!")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)

In [ ]:
!pip install -q lime

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
import numpy as np
import joblib

DATA_DIR = "data/processed/arrays_with_level"

train_data = np.load(f"{DATA_DIR}/train.npz")
test_data = np.load(f"{DATA_DIR}/test_primary.npz")

X_train = train_data["X"].squeeze(-1)
X_test = test_data["X"].squeeze(-1)
y_test = test_data["y"]

bundle = joblib.load(f"{DATA_DIR}/preprocessor.joblib")
feature_names = bundle["features"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Features:", len(feature_names))
print("LIME imported successfully!")

In [ ]:
def predict_fn(data):
    data = np.asarray(data, dtype=np.float32)

    # LIME input: (samples, 41)
    # BiLSTM input: (samples, 41, 1)
    data = data.reshape(data.shape[0], 41, 1)

    attack_prob = model.predict(data, verbose=0).reshape(-1)

    normal_prob = 1.0 - attack_prob

    return np.column_stack([normal_prob, attack_prob])


# Test prediction function
sample = X_test[:5]

predictions = predict_fn(sample)

print("Prediction probabilities:")
print(predictions)

print("\nOutput shape:", predictions.shape)
print("Probability sums:", predictions.sum(axis=1))

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

explainer = LimeTabularExplainer(
    training_data=X_train,
    feature_names=feature_names,
    class_names=["Normal", "Attack"],
    mode="classification",
    discretize_continuous=True,
    random_state=42
)

print("LIME explainer initialized successfully!")
print("Number of features:", len(feature_names))

In [ ]:
import numpy as np
from collections import Counter

DATA_DIR = "data/processed/arrays_with_level"

test_data = np.load(
    f"{DATA_DIR}/test_primary.npz",
    allow_pickle=True
)

test_labels = test_data["labels"]

print("Test label distribution:")
print(Counter(test_labels.tolist()))

rare_classes = [
    "ftp_write",
    "imap",
    "loadmodule",
    "multihop",
    "perl",
    "phf",
    "rootkit",
    "spy"
]

print("\nRare zero-day classes:")
for attack in rare_classes:
    count = np.sum(test_labels == attack)
    print(f"{attack}: {count}")

In [ ]:
import numpy as np

rare_classes = [
    "ftp_write", "imap", "loadmodule", "multihop",
    "perl", "phf", "rootkit", "spy"
]

rare_indices = np.where(np.isin(test_labels, rare_classes))[0]

print("Total zero-day samples:", len(rare_indices))

# Get predictions for zero-day samples
rare_predictions = predict_fn(X_test[rare_indices])

predicted_classes = np.argmax(rare_predictions, axis=1)

# Find correctly detected attacks
correct_indices = rare_indices[predicted_classes == 1]

print("Correctly detected zero-day attacks:", len(correct_indices))

if len(correct_indices) > 0:
    sample_idx = int(correct_indices[0])

    print("\nSelected sample index:", sample_idx)
    print("Attack type:", test_labels[sample_idx])
    print("Predicted class:", predicted_classes[
        np.where(rare_indices == sample_idx)[0][0]
    ])
    print("Attack probability:", predict_fn(X_test[sample_idx:sample_idx+1])[0][1])
else:
    print("No correctly detected zero-day samples found.")

In [ ]:
import matplotlib.pyplot as plt

lime_exp = explainer.explain_instance(
    data_row=X_test[sample_idx],
    predict_fn=predict_fn,
    num_features=15,
    num_samples=3000,
    labels=[1]
)

print("LIME explanation for:", test_labels[sample_idx])
print("\nTop 15 influential features:")

for feature, weight in lime_exp.as_list(label=1):
    print(f"{feature}: {weight:.4f}")

lime_exp.as_pyplot_figure(label=1)
plt.title("LIME Explanation - Zero-Day ftp_write Attack")
plt.tight_layout()
plt.show()

In [ ]:
import json

FAS_PATH = "experiments/phase1_reproduced/feature_attention_ranking.json"

with open(FAS_PATH, "r") as f:
    fas_data = json.load(f)

print("FAS file loaded successfully!")
print("Data type:", type(fas_data))

if isinstance(fas_data, dict):
    print("Keys:", fas_data.keys())
    print("\nPreview:")
    print(json.dumps(fas_data, indent=2)[:3000])
else:
    print(json.dumps(fas_data, indent=2)[:3000])

In [ ]:
import pandas as pd

# Top 10 LIME features by absolute contribution
lime_features = lime_exp.as_list(label=1)

lime_top10 = [
    feature.split(" ")[0]
    for feature, weight in sorted(
        lime_features,
        key=lambda x: abs(x[1]),
        reverse=True
    )[:10]
]

# Top 10 global FAS features
fas_top10 = [
    item["feature"]
    for item in fas_data[:10]
]

# Calculate overlap
common_features = sorted(set(lime_top10) & set(fas_top10))

overlap_count = len(common_features)
overlap_percentage = (overlap_count / 10) * 100

jaccard_similarity = (
    len(set(lime_top10) & set(fas_top10)) /
    len(set(lime_top10) | set(fas_top10))
)

print("LIME Top 10:")
print(lime_top10)

print("\nFAS Top 10:")
print(fas_top10)

print("\nCommon features:")
print(common_features)

print("\nTop-10 overlap:", overlap_count, "/ 10")
print("Overlap percentage:", overlap_percentage, "%")
print("Jaccard similarity:", round(jaccard_similarity, 4))

In [ ]:
# Map LIME conditions back to actual feature names
lime_conditions = lime_exp.as_list(label=1)

lime_top10_conditions = sorted(
    lime_conditions,
    key=lambda x: abs(x[1]),
    reverse=True
)[:10]

lime_top10 = []

for condition, weight in lime_top10_conditions:
    matched = [
        name for name in feature_names
        if name in condition
    ]

    if matched:
        feature = max(matched, key=len)
        lime_top10.append(feature)

fas_top10 = [item["feature"] for item in fas_data[:10]]

common_features = sorted(set(lime_top10) & set(fas_top10))

overlap_percentage = len(common_features) / 10 * 100

jaccard_similarity = (
    len(set(lime_top10) & set(fas_top10))
    / len(set(lime_top10) | set(fas_top10))
)

print("LIME Top 10:", lime_top10)
print("FAS Top 10:", fas_top10)
print("Common features:", common_features)
print("Overlap:", len(common_features), "/ 10")
print("Overlap percentage:", overlap_percentage, "%")
print("Jaccard similarity:", round(jaccard_similarity, 4))

In [ ]:
import time
import numpy as np

# 1. Retrieve local explanation fidelity score
fidelity = getattr(lime_exp, "score", None)

print("LIME local fidelity score:", fidelity)

# 2. Measure explanation generation runtime
start_time = time.perf_counter()

lime_exp_runtime = explainer.explain_instance(
    data_row=X_test[sample_idx],
    predict_fn=predict_fn,
    num_features=15,
    num_samples=3000,
    labels=[1]
)

runtime = time.perf_counter() - start_time

print("LIME explanation runtime:", round(runtime, 4), "seconds")

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

OUT_DIR = "reports/phase2_lime"
os.makedirs(OUT_DIR, exist_ok=True)

# Extract LIME explanation
lime_items = lime_exp.as_list(label=1)

lime_df = pd.DataFrame(
    lime_items,
    columns=["lime_condition", "lime_weight"]
)

# Map conditions to feature names
def map_feature(condition):
    matches = [f for f in feature_names if f in condition]
    return max(matches, key=len) if matches else condition

lime_df["feature"] = lime_df["lime_condition"].apply(map_feature)

# FAS ranking dataframe
fas_df = pd.DataFrame(fas_data)
fas_df["fas_rank"] = fas_df["rank"]

# Merge for comparison
comparison_df = lime_df.merge(
    fas_df[["feature", "rank", "mean_attention"]],
    on="feature",
    how="left"
)

comparison_df.to_csv(
    f"{OUT_DIR}/lime_fas_comparison.csv",
    index=False
)

# Save experiment metrics
results = {
    "sample_index": int(sample_idx),
    "attack_type": str(test_labels[sample_idx]),
    "attack_probability": float(predict_fn(
        X_test[sample_idx:sample_idx+1]
    )[0][1]),
    "lime_fidelity": float(fidelity),
    "lime_runtime_seconds": float(runtime),
    "top10_overlap_count": int(overlap_count),
    "top10_overlap_percentage": float(overlap_percentage),
    "jaccard_similarity": float(jaccard_similarity),
    "common_features": common_features
}

with open(f"{OUT_DIR}/experiment_metrics.json", "w") as f:
    json.dump(results, f, indent=4)

# Plot 1: LIME feature contributions
plot_df = lime_df.sort_values(
    "lime_weight",
    key=lambda x: abs(x),
    ascending=True
)

plt.figure(figsize=(10, 7))
plt.barh(plot_df["feature"], plot_df["lime_weight"])
plt.xlabel("LIME contribution weight")
plt.title("LIME Explanation - ftp_write Zero-Day Sample")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/lime_explanation.png", dpi=300)
plt.show()

# Plot 2: FAS global ranking
fas_plot = fas_df.head(10).sort_values(
    "mean_attention",
    ascending=True
)

plt.figure(figsize=(10, 7))
plt.barh(fas_plot["feature"], fas_plot["mean_attention"])
plt.xlabel("Mean attention score")
plt.title("Top 10 Global Feature Attention Scores")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fas_top10.png", dpi=300)
plt.show()

print("Results saved successfully!")
print("Output directory:", OUT_DIR)
print(json.dumps(results, indent=4))

In [ ]:
import json

metrics_path = "reports/phase2_lime/experiment_metrics.json"

with open(metrics_path, "r") as f:
    results = json.load(f)

results["top10_overlap_count"] = len(results["common_features"])
results["top10_overlap_percentage"] = (
    len(results["common_features"]) / 10 * 100
)

with open(metrics_path, "w") as f:
    json.dump(results, f, indent=4)

print(json.dumps(results, indent=4))

In [ ]:
import json
import os

metrics_path = "reports/phase2_lime/experiment_metrics.json"

with open(metrics_path, "r") as f:
    results = json.load(f)

print(json.dumps(results, indent=4))

In [ ]:
# Independently recalculate the Top-10 overlap

lime_conditions = lime_exp.as_list(label=1)

lime_top10_conditions = sorted(
    lime_conditions,
    key=lambda x: abs(x[1]),
    reverse=True
)[:10]

lime_top10_verified = []

for condition, weight in lime_top10_conditions:
    matches = [f for f in feature_names if f in condition]
    if matches:
        lime_top10_verified.append(max(matches, key=len))

fas_top10_verified = [
    item["feature"] for item in fas_data[:10]
]

lime_set = set(lime_top10_verified)
fas_set = set(fas_top10_verified)

intersection = lime_set & fas_set
union = lime_set | fas_set

print("LIME Top 10:", lime_top10_verified)
print("FAS Top 10:", fas_top10_verified)
print("Intersection:", sorted(intersection))
print("Overlap count:", len(intersection))
print("Overlap percentage:", len(intersection) / 10 * 100)
print("Jaccard similarity:", len(intersection) / len(union))

In [ ]:
import tensorflow as tf
import joblib
import numpy as np
import os

MODEL_PATH = (
    "experiments/phase1_reproduced/"
    "01_full_features_with_attention/best_model.keras"
)

DATA_DIR = "data/processed/arrays_with_level"

# Verify model checkpoint
print("Model exists:", os.path.exists(MODEL_PATH))
print("Model input shape:", model.input_shape)
print("Model output shape:", model.output_shape)

# Verify preprocessing
bundle = joblib.load(f"{DATA_DIR}/preprocessor.joblib")

print("\nPreprocessing includes level:", bundle["include_level"])
print("Number of features:", len(bundle["features"]))

# Verify test input
test_data = np.load(
    f"{DATA_DIR}/test_primary.npz",
    allow_pickle=True
)

print("\nTest X shape:", test_data["X"].shape)
print("Test y shape:", test_data["y"].shape)

print("\nFeature count matches model:",
      len(bundle["features"]) == model.input_shape[1])

In [ ]:
import numpy as np
import tensorflow as tf
import pandas as pd

# Explicitly convert and reshape test data
X_test_model = np.asarray(X_test, dtype=np.float32)

if X_test_model.ndim == 2:
    X_test_model = X_test_model.reshape(-1, 41, 1)

print("Input shape:", X_test_model.shape)
print("Input dtype:", X_test_model.dtype)

# Predict in batches using direct model calls
all_probs = []

for start in range(0, len(X_test_model), 256):
    batch = tf.convert_to_tensor(
        X_test_model[start:start + 256],
        dtype=tf.float32
    )

    batch_probs = model(batch, training=False).numpy().ravel()
    all_probs.extend(batch_probs)

test_probs = np.asarray(all_probs)
test_preds = (test_probs >= 0.5).astype(int)

print("Prediction shape:", test_probs.shape)
print("Predictions completed successfully.")

In [ ]:
import numpy as np
import pandas as pd

labels = np.asarray(test_labels).astype(str)

zero_day_classes = [
    "ftp_write", "imap", "loadmodule", "multihop",
    "perl", "phf", "rootkit", "spy"
]

rows = []

for attack in zero_day_classes:
    mask = labels == attack

    total = int(mask.sum())
    detected = int(np.sum(mask & (test_preds == 1)))
    missed = total - detected

    rows.append({
        "Attack": attack,
        "Total Samples": total,
        "Correctly Detected": detected,
        "False Negatives": missed,
        "Detection Rate (%)": round(
            100 * detected / total, 2
        ) if total > 0 else 0
    })

zero_day_summary = pd.DataFrame(rows)

display(zero_day_summary)

print("Total zero-day samples:", zero_day_summary["Total Samples"].sum())
print("Total correctly detected:", zero_day_summary["Correctly Detected"].sum())
print("Total false negatives:", zero_day_summary["False Negatives"].sum())

In [ ]:
import json

FAS_PATH = (
    "experiments/phase1_reproduced/"
    "feature_attention_ranking.json"
)

with open(FAS_PATH, "r") as f:
    fas_data = json.load(f)

print("FAS data type:", type(fas_data))

if isinstance(fas_data, dict):
    print("FAS keys:", list(fas_data.keys()))

print("\nFirst 3 records:")
print(json.dumps(fas_data[:3] if isinstance(fas_data, list)
                 else fas_data, indent=2)[:2500])

In [ ]:
import time
import numpy as np
import pandas as pd
from lime.lime_tabular import LimeTabularExplainer

# --------------------------------------------------
# 1. Prepare data and global FAS Top-10
# --------------------------------------------------

X_test_lime = np.asarray(X_test, dtype=np.float32)

if X_test_lime.ndim == 3:
    X_test_lime = X_test_lime.squeeze(-1)

labels = np.asarray(test_labels).astype(str)

fas_top10 = [
    item["feature"]
    for item in sorted(fas_data, key=lambda x: x["rank"])[:10]
]

zero_day_classes = [
    "ftp_write", "imap", "loadmodule", "multihop",
    "perl", "phf", "rootkit", "spy"
]

zero_day_indices = np.where(
    np.isin(labels, zero_day_classes)
)[0]

print("Zero-day samples:", len(zero_day_indices))
print("FAS Top-10:", fas_top10)

# --------------------------------------------------
# 2. LIME prediction function
# --------------------------------------------------

def lime_predict_fn(data):
    data = np.asarray(data, dtype=np.float32)
    data = data.reshape(-1, 41, 1)

    probabilities = model(
        data,
        training=False
    ).numpy().ravel()

    return np.column_stack([
        1 - probabilities,
        probabilities
    ])

# --------------------------------------------------
# 3. Initialize LIME explainer
# --------------------------------------------------

lime_explainer = LimeTabularExplainer(
    training_data=np.asarray(X_train, dtype=np.float32),
    feature_names=list(feature_names),
    class_names=["Normal", "Attack"],
    mode="classification",
    discretize_continuous=True,
    random_state=42
)

# --------------------------------------------------
# 4. Evaluate every zero-day sample
# --------------------------------------------------

results = []

for count, idx in enumerate(zero_day_indices, start=1):

    start_time = time.perf_counter()

    explanation = lime_explainer.explain_instance(
        data_row=X_test_lime[idx],
        predict_fn=lime_predict_fn,
        labels=(1,),
        num_features=10,
        num_samples=3000
    )

    runtime = time.perf_counter() - start_time

    lime_weights = explanation.as_list(label=1)

    # Extract feature names from LIME conditions
    lime_top10 = []

    for condition, weight in lime_weights:
        matched = [
            feature for feature in feature_names
            if feature in condition
        ]

        if matched:
            lime_top10.append(matched[0])

    lime_top10 = list(dict.fromkeys(lime_top10))[:10]

    common_features = sorted(
        set(lime_top10) & set(fas_top10)
    )

    union_features = (
        set(lime_top10) | set(fas_top10)
    )

    overlap_count = len(common_features)

    overlap_percentage = (
        overlap_count / 10 * 100
    )

    jaccard = (
        overlap_count / len(union_features)
        if union_features else 0
    )

    results.append({
        "sample_index": int(idx),
        "attack_type": labels[idx],
        "true_label": 1,
        "predicted_label": int(test_preds[idx]),
        "attack_probability": float(test_probs[idx]),
        "lime_fidelity": float(explanation.score),
        "lime_runtime_seconds": float(runtime),
        "lime_top10": lime_top10,
        "fas_top10": fas_top10,
        "common_features": common_features,
        "top10_overlap_count": overlap_count,
        "top10_overlap_percentage": overlap_percentage,
        "jaccard_similarity": jaccard
    })

    if count % 5 == 0:
        print(f"Completed {count}/{len(zero_day_indices)} samples")

lime_results_df = pd.DataFrame(results)

print("\nEvaluation completed.")
print("Total samples evaluated:", len(lime_results_df))

display(
    lime_results_df[
        [
            "sample_index",
            "attack_type",
            "predicted_label",
            "lime_fidelity",
            "lime_runtime_seconds",
            "top10_overlap_count",
            "jaccard_similarity"
        ]
    ].head(10)
)

In [ ]:
import os
import json
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Overall metrics
# --------------------------------------------------

overall_metrics = {
    "total_zero_day_samples": int(len(lime_results_df)),
    "zero_day_detection_rate_percentage": float(
        (lime_results_df["predicted_label"] == 1).mean() * 100
    ),
    "mean_lime_fidelity": float(
        lime_results_df["lime_fidelity"].mean()
    ),
    "median_lime_fidelity": float(
        lime_results_df["lime_fidelity"].median()
    ),
    "mean_lime_runtime_seconds": float(
        lime_results_df["lime_runtime_seconds"].mean()
    ),
    "mean_top10_overlap_count": float(
        lime_results_df["top10_overlap_count"].mean()
    ),
    "mean_top10_overlap_percentage": float(
        lime_results_df["top10_overlap_percentage"].mean()
    ),
    "mean_jaccard_similarity": float(
        lime_results_df["jaccard_similarity"].mean()
    ),
    "total_correctly_detected": int(
        (lime_results_df["predicted_label"] == 1).sum()
    ),
    "total_false_negatives": int(
        (lime_results_df["predicted_label"] == 0).sum()
    )
}

print("OVERALL METRICS")
print(json.dumps(overall_metrics, indent=2))

# --------------------------------------------------
# 2. Per-attack-class metrics
# --------------------------------------------------

class_metrics = (
    lime_results_df
    .groupby("attack_type")
    .agg(
        sample_count=("sample_index", "count"),
        detection_rate=("predicted_label",
                        lambda x: (x == 1).mean() * 100),
        mean_lime_fidelity=("lime_fidelity", "mean"),
        mean_runtime_seconds=("lime_runtime_seconds", "mean"),
        mean_overlap_count=("top10_overlap_count", "mean"),
        mean_overlap_percentage=("top10_overlap_percentage", "mean"),
        mean_jaccard=("jaccard_similarity", "mean")
    )
    .reset_index()
)

class_metrics = class_metrics.round(4)

print("\nCLASS-WISE METRICS")
display(class_metrics)

# --------------------------------------------------
# 3. Save results
# --------------------------------------------------

OUTPUT_DIR = "reports/phase2_lime"
os.makedirs(OUTPUT_DIR, exist_ok=True)

lime_results_df.to_csv(
    f"{OUTPUT_DIR}/lime_zero_day_results.csv",
    index=False
)

class_metrics.to_csv(
    f"{OUTPUT_DIR}/lime_classwise_metrics.csv",
    index=False
)

with open(
    f"{OUTPUT_DIR}/lime_overall_metrics.json", "w"
) as f:
    json.dump(overall_metrics, f, indent=2)

print("\nSaved:")
print(f"{OUTPUT_DIR}/lime_zero_day_results.csv")
print(f"{OUTPUT_DIR}/lime_classwise_metrics.csv")
print(f"{OUTPUT_DIR}/lime_overall_metrics.json")

In [ ]:
import matplotlib.pyplot as plt
import os

OUTPUT_DIR = "reports/phase2_lime"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------
# Graph 1: Class-wise LIME fidelity
# ------------------------------------------

plt.figure(figsize=(10, 6))

plt.bar(
    class_metrics["attack_type"],
    class_metrics["mean_lime_fidelity"]
)

plt.xlabel("Zero-Day Attack Class")
plt.ylabel("Mean LIME Fidelity")
plt.title("Class-Wise LIME Fidelity on Zero-Day Attacks")
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/classwise_lime_fidelity.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------
# Graph 2: Class-wise LIME-FAS Top-10 overlap
# ------------------------------------------

plt.figure(figsize=(10, 6))

plt.bar(
    class_metrics["attack_type"],
    class_metrics["mean_overlap_percentage"]
)

plt.xlabel("Zero-Day Attack Class")
plt.ylabel("Mean Top-10 Overlap (%)")
plt.title("Class-Wise LIME-FAS Feature Overlap")
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/classwise_lime_fas_overlap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------
# Graph 3: Class-wise detection rate
# ------------------------------------------

plt.figure(figsize=(10, 6))

plt.bar(
    class_metrics["attack_type"],
    class_metrics["detection_rate"]
)

plt.xlabel("Zero-Day Attack Class")
plt.ylabel("Detection Rate (%)")
plt.title("Zero-Day Attack Detection Rate by Class")
plt.xticks(rotation=45)
plt.ylim(0, 105)
plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/classwise_zero_day_detection.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("All three graphs saved successfully.")

In [ ]:
import numpy as np
import pandas as pd
import time
from lime.lime_tabular import LimeTabularExplainer

# Reference sample
SAMPLE_INDEX = 2294
NUM_SEEDS = [11, 22, 33, 42, 55]
NUM_SAMPLES = 3000
TOP_K = 10

sample = np.asarray(X_test_lime[SAMPLE_INDEX], dtype=np.float32)

def extract_lime_top_features(explanation, feature_names, top_k=10):
    feature_list = []

    for condition, weight in explanation.as_list(label=1):
        matches = [
            feature for feature in feature_names
            if feature in condition
        ]

        if matches:
            # Select the longest match to avoid substring ambiguity
            selected_feature = max(matches, key=len)

            if selected_feature not in feature_list:
                feature_list.append(selected_feature)

        if len(feature_list) >= top_k:
            break

    return feature_list

seed_results = []

for seed in NUM_SEEDS:

    explainer_seed = LimeTabularExplainer(
        training_data=np.asarray(X_train, dtype=np.float32),
        feature_names=list(feature_names),
        class_names=["Normal", "Attack"],
        mode="classification",
        discretize_continuous=True,
        random_state=seed
    )

    start_time = time.perf_counter()

    explanation = explainer_seed.explain_instance(
        data_row=sample,
        predict_fn=lime_predict_fn,
        labels=(1,),
        num_features=TOP_K,
        num_samples=NUM_SAMPLES
    )

    runtime = time.perf_counter() - start_time

    top_features = extract_lime_top_features(
        explanation,
        feature_names,
        TOP_K
    )

    seed_results.append({
        "seed": seed,
        "lime_fidelity": explanation.score,
        "runtime_seconds": runtime,
        "top_features": top_features
    })

    print(f"Seed {seed} completed.")

stability_df = pd.DataFrame(seed_results)

print("\nStability experiment results:")
display(stability_df)

In [ ]:
from itertools import combinations
import numpy as np
import pandas as pd
import json
import os

# Convert each Top-10 feature list into a set
feature_sets = [
    set(row["top_features"])
    for row in seed_results
]

# Pairwise Jaccard similarity
pairwise_results = []

for (i, set_a), (j, set_b) in combinations(
    enumerate(feature_sets), 2
):
    intersection = len(set_a & set_b)
    union = len(set_a | set_b)

    similarity = intersection / union if union else 0

    pairwise_results.append({
        "seed_1": NUM_SEEDS[i],
        "seed_2": NUM_SEEDS[j],
        "intersection_count": intersection,
        "jaccard_similarity": similarity
    })

pairwise_df = pd.DataFrame(pairwise_results)

mean_stability = pairwise_df["jaccard_similarity"].mean()
std_stability = pairwise_df["jaccard_similarity"].std()

print("PAIRWISE STABILITY RESULTS")
display(pairwise_df)

print("\nSUMMARY")
print("Number of seed pairs:", len(pairwise_df))
print("Mean pairwise Jaccard:", round(mean_stability, 4))
print("Std. deviation:", round(std_stability, 4))
print(
    "Mean LIME fidelity:",
    round(stability_df["lime_fidelity"].mean(), 4)
)
print(
    "Mean runtime:",
    round(stability_df["runtime_seconds"].mean(), 4),
    "seconds"
)

# Save stability results
OUTPUT_DIR = "reports/phase2_lime"
os.makedirs(OUTPUT_DIR, exist_ok=True)

pairwise_df.to_csv(
    f"{OUTPUT_DIR}/lime_stability_pairwise.csv",
    index=False
)

stability_summary = {
    "sample_index": SAMPLE_INDEX,
    "num_seeds": len(NUM_SEEDS),
    "num_seed_pairs": len(pairwise_df),
    "mean_pairwise_jaccard": float(mean_stability),
    "std_pairwise_jaccard": float(std_stability),
    "mean_lime_fidelity": float(
        stability_df["lime_fidelity"].mean()
    ),
    "mean_runtime_seconds": float(
        stability_df["runtime_seconds"].mean()
    )
}

with open(
    f"{OUTPUT_DIR}/lime_stability_summary.json", "w"
) as f:
    json.dump(stability_summary, f, indent=2)

print("\nStability artifacts saved.")

In [ ]:
import matplotlib.pyplot as plt
import os

OUTPUT_DIR = "reports/phase2_lime"
os.makedirs(OUTPUT_DIR, exist_ok=True)

pair_labels = [
    f"{row.seed_1}-{row.seed_2}"
    for row in pairwise_df.itertuples()
]

plt.figure(figsize=(10, 6))

plt.bar(
    pair_labels,
    pairwise_df["jaccard_similarity"]
)

plt.axhline(
    mean_stability,
    linestyle="--",
    label=f"Mean Stability = {mean_stability:.4f}"
)

plt.xlabel("Random Seed Pair")
plt.ylabel("Jaccard Similarity")
plt.title("LIME Explanation Stability Across Random Seeds")
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/lime_stability.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Stability graph saved.")

In [ ]:
import os
import zipfile

OUTPUT_DIR = "reports/phase2_lime"
os.makedirs("reports", exist_ok=True)

ZIP_PATH = "reports/phase2_lime_final_results.zip"

files_to_package = [
    "lime_zero_day_results.csv",
    "lime_classwise_metrics.csv",
    "lime_overall_metrics.json",
    "classwise_lime_fidelity.png",
    "classwise_lime_fas_overlap.png",
    "classwise_zero_day_detection.png",
    "lime_explanation.png",
    "fas_top10.png",
    "lime_fas_comparison.csv",
    "experiment_metrics.json",
    "lime_stability_pairwise.csv",
    "lime_stability_summary.json",
    "lime_stability.png"
]

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for filename in files_to_package:
        filepath = os.path.join(OUTPUT_DIR, filename)

        if os.path.exists(filepath):
            zipf.write(filepath, arcname=filename)
            print("Added:", filename)
        else:
            print("Missing/skipped:", filename)

print("\nUpdated ZIP:", ZIP_PATH)
print("ZIP size:", round(os.path.getsize(ZIP_PATH) / 1024, 2), "KB")

In [ ]:
from google.colab import files

files.download("reports/phase2_lime_final_results.zip")

In [ ]:
import numpy as np
import pandas as pd
import time
from itertools import combinations
from lime.lime_tabular import LimeTabularExplainer

# --------------------------------------------------
# Configuration
# --------------------------------------------------

ATTACK_CLASSES = [
    "ftp_write", "imap", "loadmodule", "multihop",
    "perl", "phf", "rootkit", "spy"
]

STABILITY_SEEDS = [11, 42, 55]
SAMPLES_PER_CLASS = 2
NUM_LIME_SAMPLES = 3000
TOP_K = 10

# --------------------------------------------------
# Select up to 2 representative samples per class
# --------------------------------------------------

rng = np.random.default_rng(42)

selected_samples = []

for attack in ATTACK_CLASSES:
    class_indices = np.where(labels == attack)[0]

    chosen = rng.choice(
        class_indices,
        size=min(SAMPLES_PER_CLASS, len(class_indices)),
        replace=False
    )

    selected_samples.extend(chosen.tolist())

print("Selected samples:", len(selected_samples))

# --------------------------------------------------
# Evaluate stability
# --------------------------------------------------

multi_sample_stability = []
multi_sample_pairwise = []

for sample_idx in selected_samples:

    sample_features = X_test_lime[sample_idx]
    attack_type = labels[sample_idx]

    seed_feature_sets = []

    for seed in STABILITY_SEEDS:

        explainer = LimeTabularExplainer(
            training_data=np.asarray(X_train, dtype=np.float32),
            feature_names=list(feature_names),
            class_names=["Normal", "Attack"],
            mode="classification",
            discretize_continuous=True,
            random_state=seed
        )

        start = time.perf_counter()

        explanation = explainer.explain_instance(
            data_row=sample_features,
            predict_fn=lime_predict_fn,
            labels=(1,),
            num_features=TOP_K,
            num_samples=NUM_LIME_SAMPLES
        )

        runtime = time.perf_counter() - start

        top_features = extract_lime_top_features(
            explanation,
            feature_names,
            TOP_K
        )

        seed_feature_sets.append(set(top_features))

        multi_sample_stability.append({
            "sample_index": int(sample_idx),
            "attack_type": attack_type,
            "seed": seed,
            "lime_fidelity": float(explanation.score),
            "runtime_seconds": float(runtime),
            "top_features": top_features
        })

    # Pairwise Jaccard for this sample
    for (i, set_a), (j, set_b) in combinations(
        enumerate(seed_feature_sets), 2
    ):
        intersection = len(set_a & set_b)
        union = len(set_a | set_b)

        jaccard = intersection / union if union else 0

        multi_sample_pairwise.append({
            "sample_index": int(sample_idx),
            "attack_type": attack_type,
            "seed_1": STABILITY_SEEDS[i],
            "seed_2": STABILITY_SEEDS[j],
            "intersection_count": intersection,
            "jaccard_similarity": jaccard
        })

    print(f"Completed sample {sample_idx} ({attack_type})")

# --------------------------------------------------
# Summarize
# --------------------------------------------------

multi_stability_df = pd.DataFrame(multi_sample_stability)
multi_pairwise_df = pd.DataFrame(multi_sample_pairwise)

print("\nMean multi-sample stability:",
      multi_pairwise_df["jaccard_similarity"].mean())

print("Standard deviation:",
      multi_pairwise_df["jaccard_similarity"].std())

print("\nClass-wise stability:")
display(
    multi_pairwise_df
    .groupby("attack_type")["jaccard_similarity"]
    .agg(["mean", "std", "count"])
    .round(4)
)

# --------------------------------------------------
# Save results
# --------------------------------------------------

import os

OUTPUT_DIR = "reports/phase2_lime"
os.makedirs(OUTPUT_DIR, exist_ok=True)

multi_stability_df.to_csv(
    f"{OUTPUT_DIR}/lime_multisample_stability.csv",
    index=False
)

multi_pairwise_df.to_csv(
    f"{OUTPUT_DIR}/lime_multisample_pairwise.csv",
    index=False
)

print("\nMulti-sample stability results saved.")